#### 공공 자전거 대여 데이터 파이프라인
**STEP 1 · 진단** — 1-1. 원본 그대로 읽기

In [6]:
import pandas as pd

stations = pd.read_csv("data/stations.csv")
bikes    = pd.read_csv("data/raw-bikes.csv")
rentals  = pd.read_csv("data/raw-rentals.csv")

print("[stations]")
print(stations.dtypes)
print()
print("[bikes]")
print(bikes.dtypes)
print()
print("[rentals]")
print(rentals.dtypes)

[stations]
station_id      str
station_name    str
district        str
dtype: object

[bikes]
bike_id             str
station_id          str
bike_type           str
gear_count          str
manufacture_year    str
daily_fee           str
dtype: object

[rentals]
rental_id             str
bike_id               str
user_id               str
rent_time             str
return_time           str
distance_km       float64
fee                   str
payment_method        str
dtype: object


**1-3. 숫자여야 하는데 문자열로 읽힌 컬럼**


In [7]:
# 숫자여야 하는 컬럼만 골라서 dtype 확인
num_cols_bikes   = ["gear_count", "manufacture_year", "daily_fee"]
num_cols_rentals = ["distance_km", "fee"]

print("[bikes]")
print(bikes[num_cols_bikes].dtypes)
print()
print("[rentals]")
print(rentals[num_cols_rentals].dtypes)

[bikes]
gear_count          str
manufacture_year    str
daily_fee           str
dtype: object

[rentals]
distance_km    float64
fee                str
dtype: object


→ 문자열로 읽힌 숫자 컬럼: `gear_count`, `manufacture_year`, `daily_fee`, `fee` (4개)

→ `distance_km`은 float64지만, "N/A" 같은 값이 자동으로 NaN이 된 것이라 원본 확인 필요

In [ ]:
rentals = pd.read_csv("data/raw-rentals.csv")

missing = pd.DataFrame({
    "결측수": rentals.isna().sum(),
    "비율(%)": (rentals.isna().mean() * 100).round(2)
})
print("[rentals 결측]")
print(missing)

[rentals 결측]
                결측수  비율(%)
rental_id         0   0.00
bike_id           0   0.00
user_id           0   0.00
rent_time         0   0.00
return_time       0   0.00
distance_km     212   1.41
fee               0   0.00
payment_method    0   0.00


**1-5. distance_km 을 숫자로 못 바꾸는 값**

In [ ]:
rentals_raw = pd.read_csv("data/raw-rentals.csv", dtype=str, keep_default_na=False)

converted = pd.to_numeric(rentals_raw["distance_km"], errors="coerce")
bad = rentals_raw.loc[converted.isna(), "distance_km"]

print("변환 실패 건수:", len(bad))
print(bad.value_counts())

변환 실패 건수: 212
distance_km
N/A    212
Name: count, dtype: int64


**1-6. bike_id, rental_id 중복 건수**

In [10]:
print("bike_id 중복:", bikes_raw["bike_id"].duplicated().sum())
print("rental_id 중복:", rentals_raw["rental_id"].duplicated().sum())

bike_id 중복: 5
rental_id 중복: 500


→ bike_id 중복 5건 → 실제 자전거 50대 / rental_id 중복 500건 (배치 스크립트 오작동)

**1-7. bike_type, district, payment_method 고유값**

In [11]:
print("[bike_type]", bikes_raw["bike_type"].nunique(), "종류")
print(bikes_raw["bike_type"].unique())
print()
print("[district]", stations_raw["district"].nunique(), "종류")
print(stations_raw["district"].unique())
print()
print("[payment_method]", rentals_raw["payment_method"].nunique(), "종류")
print(rentals_raw["payment_method"].unique())

[bike_type] 6 종류
<StringArray>
['일반　', '일 반', '일반', '전동', 'electric', '일반형']
Length: 6, dtype: str

[district] 10 종류
<StringArray>
['강남구', '강동구', '강북구', '강서구', '관악구', '광진구', '구로구', '금천구', '노원구', '도봉구']
Length: 10, dtype: str

[payment_method] 13 종류
<StringArray>
[      'CARD',       ' APP', 'membership', 'Membership',       'APP ',
        'app',      'CARD ', 'MEMBERSHIP',        'App',      ' CARD',
       'card',       'Card',        'APP']
Length: 13, dtype: str


**1-8. bike_type 실제 종류 판단**

→ bike_type 고유값 6종 → 실제는 2종 (일반 / 전동)
  - 일반: '일반　'(전각 공백), '일 반', '일반', '일반형'
  - 전동: '전동', 'electric'

**1-9. bikes 숫자 컬럼 원본 값 확인**

In [13]:
for col in ["station_id","gear_count","manufacture_year", "daily_fee"]:
    print(f"[{col}]")
    print(bikes_raw[col].value_counts())
    print()

[station_id]
station_id
S23    4
S08    4
S13    3
S25    3
S03    3
S12    3
S02    3
s11    2
S05    2
S20    2
S30    2
S15    2
S24    2
S18    2
S06    2
S19    1
S26    1
s22    1
S10    1
S22    1
s23    1
s18    1
S28    1
S21    1
       1
s14    1
s26    1
S04    1
S27    1
S01    1
s03    1
Name: count, dtype: int64

[gear_count]
gear_count
3단    25
３단    20
7단     7
７단     3
Name: count, dtype: int64

[manufacture_year]
manufacture_year
2018    11
2016     8
2019     6
N/A      6
2015     6
2020     6
2023     5
2017     2
불명       2
2021     2
2022     1
Name: count, dtype: int64

[daily_fee]
daily_fee
1,000    45
2,000    10
Name: count, dtype: int64



**STEP 1 진단 결과 요약**

| 파일 | 컬럼 | 문제 | 건수 |
|---|---|---|---|
| bikes | bike_id | 중복 | 5 |
| bikes | station_id | 대소문자 혼재(S11/s11), 빈 값 | 빈 값 1 |
| bikes | bike_type | 6종 혼재(전각 공백, 가운데 공백, electric, 일반형) → 실제 2종 | - |
| bikes | gear_count | "단" 단위 포함, 전각 숫자(３단) | 전부 문자열 |
| bikes | manufacture_year | "N/A", "불명" | 8 |
| bikes | daily_fee | 콤마 포함(1,000) | 전부 문자열 |
| rentals | rental_id | 중복 | 500 |
| rentals | distance_km | "N/A" | 212 |
| rentals | fee | 문자열로 읽힘 | - |
| rentals | payment_method | 앞뒤 공백, 대소문자 혼재 → 실제 3종 | 13종 |
| stations | - | 깨끗함 | - |

#### STEP 2 · 자전거 마스터 정제
**2-1. 전각 문자 → 반각으로 통일**

In [16]:
import unicodedata

def clean_text(s):
    s = unicodedata.normalize("NFKC",s)
    s = s.strip()
    return s

print(repr(clean_text("3단")))
print(repr(clean_text("일반 ")))

'3단'
'일반'


**2-2. 전체 컬럼에 clean_text 적용**

In [18]:
bikes = bikes_raw.copy()

for col in bikes.columns:
    bikes[col] = bikes[col].map(clean_text)

print(bikes["gear_count"].value_counts())    

gear_count
3단    45
7단    10
Name: count, dtype: int64


**2-3. station_id — 대문자 통일, 빈 값은 결측**

In [22]:
bikes["station_id"] = bikes["station_id"].str.upper()
bikes["station_id"] = bikes["station_id"].replace("", pd.NA)

print("고유값 수:", bikes["station_id"].nunique())
print("결측 수:", bikes["station_id"].isna().sum())

고유값 수: 25
결측 수: 1


**2-4. bike_type — 일반 / 전동 두 가지로 통일**

In [23]:
bikes["bike_type"] = bikes["bike_type"].replace({
    "일 반":  "일반",
    "일반형": "일반",
    "electric": "전동",
})

print(bikes["bike_type"].unique())

<StringArray>
['일반', '전동']
Length: 2, dtype: str


**2-5. gear_count — 단위 문자 제거 후 정수로**

In [24]:
bikes["gear_count"] = bikes["gear_count"].str.replace("단","").astype(int)

print(bikes["gear_count"].dtype)
print(bikes["gear_count"].min(),"~",bikes["gear_count"].max())

int64
3 ~ 7


**2-6. daily_fee — 콤마 제거 후 정수로**

In [25]:
bikes["daily_fee"] = bikes["daily_fee"].str.replace(",","").astype(int)

print(bikes["daily_fee"].dtype)
print(bikes["daily_fee"].min(), "~", bikes["daily_fee"].max())

int64
1000 ~ 2000


**2-7. manufacture_year — 정수로 (변환 실패는 결측)**

In [27]:
bikes["manufacture_year"] = pd.to_numeric(bikes["manufacture_year"], errors="coerce").astype("Int64")

print(bikes["manufacture_year"].dtype)
print("결측 수:", bikes["manufacture_year"].isna().sum())
print(bikes["manufacture_year"].min(), "~", bikes["manufacture_year"].max())

Int64
결측 수: 8
2015 ~ 2023
